# Data processing 
In this notebook you find examples of how processing works, including
- Preprocessing transliteration
- Preprocessing translation
- Postprocessing translation
- Preprocessing of published.csv transliteration
- Full example on a real sentence from the train set

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import re

import pandas as pd
from data_processing.processing import (
    PREPROCESSING_CONFIG,
    TextProcessor
)
from data_processing.augmentation import DataAugmentation

tp = TextProcessor()

## TRANSLITERATION PREPROCESSING
Below are examples of all the elements preprocessed in transliteration

In [2]:
print("HANDLING LOGOGRAMS\n")
text = "KÙ.B. ni-is-ḫa-sú DIRI na ŠÀ.BA-ta"
print("In:\n    ", text)
print("Out: ")
print("   with separation: ", tp.preprocess_transliteration_text(text, separate_compounds=True))
print("   no separation:   ", tp.preprocess_transliteration_text(text, separate_compounds=False))


HANDLING LOGOGRAMS

In:
     KÙ.B. ni-is-ḫa-sú DIRI na ŠÀ.BA-ta
Out: 
   with separation:  <lgg_s>kù<lgg_e> <lgg_s>babbar<lgg_e> niisḫasú <lgg_s>diri<lgg_e> na <lgg_s>šà<lgg_e> <lgg_s>ba<lgg_e>ta
   no separation:    <lgg_s>kù.babbar<lgg_e> niisḫasú <lgg_s>diri<lgg_e> na <lgg_s>šà.ba<lgg_e>ta


In [3]:
print("HANDLING SUBSCRIPTS AND ACCENTS \n")
text = "tur₄ im-ḫu-kà ù ša eš"
print("In:\n    ", text)
print("Out: \n")
print("    Accent normalization:   ", tp.preprocess_transliteration_text(text, normalize_chars=True))
print("    No accent normalization:", tp.preprocess_transliteration_text(text, normalize_chars=False))

HANDLING SUBSCRIPTS AND ACCENTS 

In:
     tur₄ im-ḫu-kà ù ša eš
Out: 

    Accent normalization:    tur4 imhuka3 u3 sza esz
    No accent normalization: tur₄ imḫukà ù ša eš


In [4]:
print("HANDLING GAPS \n")
text = "a šé-né-šu <gap> (broken line). And <gap>-<gap> <gap> <gap>, <gap> I gave it to U-<gap>"
print("In:\n    ", text)
print("Out: \n")
print("   ", tp.preprocess_transliteration_text(text))

HANDLING GAPS 

In:
     a šé-né-šu <gap> (broken line). And <gap>-<gap> <gap> <gap>, <gap> I gave it to U-<gap>
Out: 

    a šénéšu <gap> broken line. and <gap> , <gap> <lgg_s>i<lgg_e> gave it to <lgg_s>u<lgg_e><gap>


In [5]:
print("HANDLING HYPHENS \n")
text = "šé-né-šu šál-ma-a-šùr"
print("In:\n    ", text)
print("Out: \n")
print("   With hyphens:   ", tp.preprocess_transliteration_text(text, with_hyphens=True))
print("   Without hyphens:", tp.preprocess_transliteration_text(text, with_hyphens=False))

HANDLING HYPHENS 

In:
     šé-né-šu šál-ma-a-šùr
Out: 

   With hyphens:    šé-né-šu šál-ma-a-šùr
   Without hyphens: šénéšu šálmaašùr


In [6]:
print("HANDLING DETERMINATIVES \n")
text = "šu-{d}DUMU.ZIa-lim{ki} a{unexistent}, ana{giš} {mul}-ma-ni{é}"
print("In:\n    ", text)
print("Out: \n")
print("   Named:   ", tp.preprocess_transliteration_text(text, named_determinatives=True))
print("   Akkadian:", tp.preprocess_transliteration_text(text, named_determinatives=False))

HANDLING DETERMINATIVES 

In:
     šu-{d}DUMU.ZIa-lim{ki} a{unexistent}, ana{giš} {mul}-ma-ni{é}
Out: 

   Named:    šu- god <lgg_s>dumu<lgg_e>.zialim earth a{unexistent}, ana wood stars -mani building
   Akkadian: šu-{d}<lgg_s>dumu<lgg_e>.zialim{ki} a{unexistent}, ana{giš} {mul}-mani{é}


## TRANSLATION PREPROCESSING
Below are examples of all the elements preprocessed in transliteration

In [7]:
print("HANDLING NUMERALS \n\n")
print("------------------Roman numerals:\n")
text = "In month I and in month II I will go"
print("In:\n    ", text)
print("Out: \n")
print("    ", tp.preprocess_translation_text(text), "\n\n")

print("------------------Fractions and decimals: \n")
text = ("I need 0.5 or 0.333 shikels and a fourth of the grains")
print("In:\n    ", text)
print("Out: \n")
print("    ", tp.preprocess_translation_text(text), "\n\n")

print("------------------Special amounts: \n")
text = ("took 5 11 / 12 shekels")
print("In:\n    ", text)
print("Out: \n")
print(tp.preprocess_translation_text(text), "\n\n")

HANDLING NUMERALS 


------------------Roman numerals:

In:
     In month I and in month II I will go
Out: 

     In month 1 and in month 2 I will go 


------------------Fractions and decimals: 

In:
     I need 0.5 or 0.333 shikels and a fourth of the grains
Out: 

     I need ½ or ⅓ shikels and a fourth of the grains 


------------------Special amounts: 

In:
     took 5 11 / 12 shekels
Out: 

took 6 shekels less 15 grains 




In [8]:
print("HANDLING ANNOTATIOS \n\n")

text = "Why is that you (fem. plur.) have written me"
print("In:\n    ", text)
print("Out: \n")
print(tp.preprocess_translation_text(text), "\n\n")

HANDLING ANNOTATIOS 


In:
     Why is that you (fem. plur.) have written me
Out: 

Why is that you have written me 




In [9]:
print("EXPANDING APPBREVIATIONS")
text = ("Give me gold and -gold and don't pay -tax nor road-tax but textile")
print("In:\n    ", text)
print("Out: \n")
print("    ", tp.preprocess_translation_text(text), "\n\n")

EXPANDING APPBREVIATIONS
In:
     Give me gold and -gold and don't pay -tax nor road-tax but textile
Out: 

     Give me gold and pašallum gold and don't pay šadduātum tax nor road-tax but textile 




In [10]:
print("CLEANUPS")
text = ("You (sing.)      are in love with her/him ) ")
print("In:\n    ", text)
print("Out: \n")
print("    ", tp.preprocess_translation_text(text), "\n\n")

CLEANUPS
In:
     You (sing.)      are in love with her/him ) 
Out: 

     You are in love with her ) 




In [11]:
print("HANDLING QUESTION MARKS")
print("---------------SUCCESSFUL EXAMPLE")
translation_text = "Lamassutum (and?), How are you? (?)"
print(tp.preprocess_translation_text(translation_text), "\n\n")

HANDLING QUESTION MARKS
---------------SUCCESSFUL EXAMPLE
Lamassutum (and?), How are you? 




## POSTPROCESSING BY CASE
Below are examples of all the elements postprocessing in transaltion output

In [12]:
print("CLEANUPS")
print("------------------REPETITION HALLUCINATIONS")
text = ("I am, I am, I am the best,,, bye bye bye          bye...,")
print("In:\n    ", text)
print("Out: \n   ")
print(tp.postprocess_translation_output(text), "\n\n")

print("------------------NUMBERS AND GAPS")
text = ("I want month I <gap>-<gap>, some one said 0.5")
print("In:\n    ", text)
print("Out: \n    ")
print(tp.postprocess_translation_output(text), "\n\n")

CLEANUPS
------------------REPETITION HALLUCINATIONS
In:
     I am, I am, I am the best,,, bye bye bye          bye...,
Out: 
   
I am the best, bye<gap>, 


------------------NUMBERS AND GAPS
In:
     I want month I <gap>-<gap>, some one said 0.5
Out: 
    
I want month 1 <gap>, some one said ½ 




# TRANSITERATION PREPROCESSING OF PUBLICATIONS CSVs

In [13]:
print("HANDLING LOGOGRAMS\n")
text = "_ku3-babbar?_ and _gin2 ku3-babbar-ta_"
print("In:\n    ", text)
print("Out: ")
print("   with separation: ", tp.preprocess_transliteration_text(text, separate_compounds=True))
print("   no separation:   ", tp.preprocess_transliteration_text(text, separate_compounds=False))


HANDLING LOGOGRAMS

In:
     _ku3-babbar?_ and _gin2 ku3-babbar-ta_
Out: 
   with separation:  <lgg_s>kù<lgg_e> <lgg_s>babbar<lgg_e> and gin2 <lgg_s>kù<lgg_e> <lgg_s>babbar<lgg_e>ta
   no separation:    <lgg_s>kù.babbar<lgg_e> and gin2 <lgg_s>kù.babbar<lgg_e>ta


In [14]:
print("HANDLING MODIFIERS\n")
text = "ta la2_ 2(asz@t)? [...] [x] x i-na-a"
print("In:\n    ", text)
print("Out: ")
print("   ", tp.preprocess_transliteration_text(text, normalize_chars=True, with_hyphens=True))


HANDLING MODIFIERS

In:
     ta la2_ 2(asz@t)? [...] [x] x i-na-a
Out: 
    ta la2_ 2asz <gap> x i-na-a


In [15]:
print("HANDLING WEIRD CHARS\n")
text = "i-di2-su2#-in# _dumu?_ szal2-mah!  i-[di2-a-szur] a-al#-du10..."
print("In:\n    ", text)
print("Out: ")
print("   ", tp.preprocess_transliteration_text(text, normalize_chars=True, with_hyphens=True))


HANDLING WEIRD CHARS

In:
     i-di2-su2#-in# _dumu?_ szal2-mah!  i-[di2-a-szur] a-al#-du10...
Out: 
    i-di2-su2-in <lgg_s>dumu<lgg_e> szal2-mah i-di2-a-szur a-al-du10<gap>


# EXAMPLE ON REAL TRAIN SET SENTENCE

In [16]:
# REAL SENTENCES TEST

test_transliteration = "a-na É šál-ma-a-šùr a-na a-wa-tim né-ru-ub-ma um-ma šál-ma-a-šùr-ma i-a-tí a-na bu-ru-uš-ḫa-dim i-na ma-ak-sú-e-im a-na ḫu-bu-ul ÌR a-bu-ša-lim ša-qá-lim i-ra-de₈-ú-ni a-ta 30 TÚG ku-ta-ni ù ANŠE ṣa-lá-ma-am ša ÌR ta-lá-qé-e um-ma šu-be-lúm-ma ke-na ša ÌR-dí-a a-bu-ša-lim al-qé ša-lim-a-šur ša ni-iš a-lim{ki} úz-na-tí-ni il₅-pu-ut um-ma ša-lim-a-šur-ma a-šar ÌR a-na ÌR-dí-kà lá i-tù-ru-ú ma-a a-na ÌR-dí-a i-tù-wa-ar um-ma šu-be-lúm-ma šu-ma a-na ku-a-im i-tù-a-ar a-na TÚG.ḪI.A ù ANŠE.ḪI.A a-za-za-kum a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-dí-ni-a-tí-ma IGI GÍR ša a-šùr ší-bu-tí-ni ni-dí-in IGI ú-ra-a IGI ì-lí-dan IGI ma-ni-um-ba-lúm-a-šùr," 
test_translation = "We entered Šalim-Aššur's house to settle the case, and Šalim-Aššur said: While they are going to lead me personally to Burušhaddum in fetters to pay the debt of the slave Abu-šalim, will you on the other hand take 30 -textiles and a black donkey from the slave?""Šu-Bēlum answered: ""Yes, I have taken what belongs to my slave Abu-šalim."" Šalim-Aššur impressed on us what had been sworn by the City, and Šalim-Aššur said: ""In case the slave does not raise claim against your slave, then, will he raise claims against my slave?"" Šu-Bēlum answered: ""If he raises claim against yours, then I guarantee for the textiles and the donkey for you."" The Kanesh colony gave us for these proceedings, and we gave our testimony before Aššur's dagger. Witnessed by Uraya, by Ilī-dan, by Mannum-balum-Aššu"
 
def run_tests(test_translation, test_transliteration):
    tp = TextProcessor()
    
    print("#"*30)
    print("1. TRANSLITERATION PREPROCESSING TESTS")
    print("#"*30)
    
    processed = tp.preprocess_transliteration_text(test_transliteration, named_determinatives=False, with_hyphens=False,normalize_chars=False, separate_compounds=False)
    print(f"In:  {test_transliteration}\nOut: {processed}\n")

    print("\n" + "#"*30)
    print("2. TRANSLATION PREPROCESSING TESTS")
    print("#"*30)
    processed = tp.preprocess_translation_text(test_translation)
    print(f"In:  {test_translation}\nOut: {processed}\n")

run_tests(test_translation, test_transliteration)

##############################
1. TRANSLITERATION PREPROCESSING TESTS
##############################
In:  a-na É šál-ma-a-šùr a-na a-wa-tim né-ru-ub-ma um-ma šál-ma-a-šùr-ma i-a-tí a-na bu-ru-uš-ḫa-dim i-na ma-ak-sú-e-im a-na ḫu-bu-ul ÌR a-bu-ša-lim ša-qá-lim i-ra-de₈-ú-ni a-ta 30 TÚG ku-ta-ni ù ANŠE ṣa-lá-ma-am ša ÌR ta-lá-qé-e um-ma šu-be-lúm-ma ke-na ša ÌR-dí-a a-bu-ša-lim al-qé ša-lim-a-šur ša ni-iš a-lim{ki} úz-na-tí-ni il₅-pu-ut um-ma ša-lim-a-šur-ma a-šar ÌR a-na ÌR-dí-kà lá i-tù-ru-ú ma-a a-na ÌR-dí-a i-tù-wa-ar um-ma šu-be-lúm-ma šu-ma a-na ku-a-im i-tù-a-ar a-na TÚG.ḪI.A ù ANŠE.ḪI.A a-za-za-kum a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-dí-ni-a-tí-ma IGI GÍR ša a-šùr ší-bu-tí-ni ni-dí-in IGI ú-ra-a IGI ì-lí-dan IGI ma-ni-um-ba-lúm-a-šùr,
Out: ana <lgg_s>é<lgg_e> šálmaašùr ana awatim néruubma umma šálmaašùrma iatí ana buruušḫadim ina maaksúeim ana ḫubuul <lgg_s>ìr<lgg_e> abušalim šaqálim irade₈-úni ata 30 <lgg_s>túg<lgg_e> kutani ù <lgg_s>anše<lgg_e> ṣalámaam ša <lgg_s>ìr<lg

In [ ]:
# REAL SENTENCES TEST from published.csv

test_transliteration = "um-ma wa-ak-lum2-ma a-na# [ka3]-ri-im ka3-ni-isz(ki) qi2-bi2#-ma# a-lu-um [di2]-nam# _igi_ 2(disz) _gesz-gag-en [gal?_ i]-na ha-am-ri-im [i-di2-in]-ma# 5(disz) a-wi-li [...] x"
test_translation = "Thus ""Foreman,"" to Karum-Kanesh: speak! The city a verdict, before two shugarria'um-symbols, in the sacred precinct gave; five men ..., ...,"
 
def run_tests(test_translation, test_transliteration):
    tp = TextProcessor()
    
    print("#"*30)
    print("1. TRANSLITERATION PREPROCESSING TESTS")
    print("#"*30)
    
    processed = tp.preprocess_transliteration_text(test_transliteration, named_determinatives=False, with_hyphens=False,normalize_chars=False, separate_compounds=False)
    print(f"In:  {test_transliteration}\nOut: {processed}\n")

    print("\n" + "#"*30)
    print("2. TRANSLATION PREPROCESSING TESTS")
    print("#"*30)
    processed = tp.preprocess_translation_text(test_translation)
    print(f"In:  {test_translation}\nOut: {processed}\n")

run_tests(test_translation, test_transliteration)

##############################
1. TRANSLITERATION PREPROCESSING TESTS
##############################
In:  um-ma wa-ak-lum2-ma a-na# [ka3]-ri-im ka3-ni-isz(ki) qi2-bi2#-ma# a-lu-um [di2]-nam# _igi_ 2(disz) _gesz-gag-en [gal?_ i]-na ha-am-ri-im [i-di2-in]-ma# 5(disz) a-wi-li [...] x
Out: umma waaklum2ma ana ka3riim ka3niiszki qi2bi2ma aluum di2nam <lgg_s>igi<lgg_e> 2disz <lgg_s>gesz<lgg_e><lgg_s>gag<lgg_e><lgg_s>en<lgg_e> <lgg_s>gal<lgg_e> ina haamriim idi2inma 5disz awili <gap>


##############################
2. TRANSLATION PREPROCESSING TESTS
##############################
In:  Thusx Foreman, to Karum-Kanesh: speak! The city a verdict, before two shugarria'um-symbols, in the sacred precinct gave; five men ..., ...,
Out: Thus Foreman, to Karum-Kanesh: speak! The city a verdict, before 2 shugarria'um-symbols, in the sacred precinct gave; 5 men,,

